# 03 — Auto-encodeur convolutionnel et temporalité

Mode utilisé dans ce support : **smoke**, destiné à vérifier la chaîne complète. Les valeurs commentées proviennent du rapport calculé, jamais de scores inventés.

In [ ]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
REPORT = ROOT / "outputs" / "reports" / "results_smoke.json"
if not REPORT.exists():
    raise FileNotFoundError("Exécutez d'abord : .venv\\Scripts\\python scripts\\run_smoke.py --mode smoke")
results = json.loads(REPORT.read_text(encoding="utf-8"))
sns.set_theme(style="whitegrid")

## Question étudiée et notions

Sur des semaines de 336 points, un CNN 1D partage un filtre le long du temps :

$$h_{c,t}=\sigma\!\left(b_c+\sum_{c',j}w_{c,c',j}x_{c',t+j}\right).$$

Les strides compressent la longueur ; trois blocs de stride 2 donnent un champ réceptif croissant. Les convolutions transposées reconstruisent puis le module tronque explicitement à 336 points. Ce partage réduit les paramètres, mais ne rend pas le réseau invariant à l'heure ni insensible aux décalages.

In [ ]:
weekly=pd.DataFrame(results['weekly']).T
weekly['temps_s']=weekly.apply(lambda r:r.get('history',{}).get('seconds') if isinstance(r.get('history'),dict) else np.nan,axis=1)
display(weekly[['rmse_observed','mae_observed','parameters','temps_s']])
weekly.rmse_observed.plot.bar(figsize=(7,3)); plt.ylabel('RMSE test observée'); plt.title('Mêmes semaines et même dimension latente'); plt.show()

In [ ]:
import joblib, torch
from cer_ae.models import ConvAutoencoder1D
from cer_ae.training import predict
w=np.load(ROOT/'data/processed/smoke_weekly_windows.npz')['values']; scaler=joblib.load(ROOT/'outputs/models/weekly_scaler.joblib')
wx=scaler.transform(w.reshape(-1,1)).reshape(w.shape).astype('float32'); dim=min(map(int,[k.split('_')[-1] for k in results['daily'] if k.startswith('pca_')]))
cnn=ConvAutoencoder1D(336,dim); cnn.load_state_dict(torch.load(ROOT/f'outputs/models/weekly_cnn1d_latent{dim}.pt',map_location='cpu',weights_only=True)); rec=predict(cnn,wx[:2])
fig,axes=plt.subplots(2,1,figsize=(12,5),sharex=True)
for ax,a,b in zip(axes,wx[:2],rec): ax.plot(a,label='semaine'); ax.plot(b,label='CNN'); [ax.axvline(d*48,color='grey',lw=.5) for d in range(1,7)]; ax.legend()
axes[-1].set_xlabel('Créneau demi-horaire (traits = jours)'); fig.supylabel('Valeur standardisée'); plt.show()

## Interprétation et annualité

Dans l'exécution courte, l'ACP est encore la meilleure reconstruction. Le CNN utilise nettement moins de paramètres que le dense hebdomadaire, mais deux époques ne suffisent pas à exploiter son biais inductif.

Une entrée d'une semaine ne peut pas apprendre directement une dynamique annuelle. L'expérience annuelle doit être distincte, sur les séries quotidiennes par foyer et une période commune longue ; elle est préparée par `df_jour`, mais n'a pas été entraînée en mode smoke.

Questions : quel est le champ réceptif exact ? Pourquoi comparer les pics séparément de la RMSE globale ? Comment encoder explicitement l'heure si les décalages sont indésirables ?